# Sistema de Gestion de Emergencias Hospitalarias

In [1]:
# Grafo aciclico dirigido (DAG) del proceso de emergencias
class GrafoAciclico:

    def __init__(self):
        self.adyacencia = {}

    def agregar_nodo(self, nodo):
        if nodo not in self.adyacencia:
            self.adyacencia[nodo] = []

    def agregar_arista(self, origen, destino):
        self.agregar_nodo(origen)
        self.agregar_nodo(destino)
        if destino not in self.adyacencia[origen]:
            self.adyacencia[origen].append(destino)

    def buscar_nodo(self, nodo):
        return nodo in self.adyacencia

    def eliminar_nodo(self, nodo):
        if nodo not in self.adyacencia:
            return False
        del self.adyacencia[nodo]
        for sucesores in self.adyacencia.values():
            if nodo in sucesores:
                sucesores.remove(nodo)
        return True

    def obtener_nodos(self):
        return list(self.adyacencia.keys())

    def obtener_sucesores(self, nodo):
        return self.adyacencia.get(nodo, [])

    def mostrar_grafo(self):
        print("Estructura del grafo (lista de adyacencia):")
        for nodo, sucesores in self.adyacencia.items():
            if sucesores:
                print(f"  {nodo} -> {', '.join(str(s) for s in sucesores)}")
            else:
                print(f"  {nodo} -> (sin dependientes)")


In [2]:
# Ordenamiento topologico algoritmo de Kahn
from collections import deque
def ordenamiento_topologico(grafo):
    nodos = grafo.obtener_nodos()
    grado_entrada = {nodo: 0 for nodo in nodos}
    for nodo in nodos:
        for sucesor in grafo.obtener_sucesores(nodo):
            grado_entrada[sucesor] += 1

    cola = deque([nodo for nodo in nodos if grado_entrada[nodo] == 0])
    orden = []

    while cola:
        actual = cola.popleft()
        orden.append(actual)
        for sucesor in grafo.obtener_sucesores(actual):
            grado_entrada[sucesor] -= 1
            if grado_entrada[sucesor] == 0:
                cola.append(sucesor)

    es_valido = len(orden) == len(nodos)
    return orden, es_valido


In [3]:
# Cola de pacientes, lista enlazada
class NodoCola:
    def __init__(self, dato):
        self.dato = dato
        self.siguiente = None

class Cola:
    def __init__(self):
        self.frente = None
        self.final = None
        self.tamano = 0

    def esta_vacia(self):
        return self.frente is None

    def encolar(self, dato):
        nuevo_nodo = NodoCola(dato)
        if self.esta_vacia():
            self.frente = nuevo_nodo
            self.final = nuevo_nodo
        else:
            self.final.siguiente = nuevo_nodo
            self.final = nuevo_nodo
        self.tamano += 1

    def desencolar(self):
        if self.esta_vacia():
            return None
        nodo_saliente = self.frente
        self.frente = self.frente.siguiente
        if self.frente is None:
            self.final = None
        self.tamano -= 1
        return nodo_saliente.dato

    def buscar_por_id(self, id_paciente):
        actual = self.frente
        while actual is not None:
            if actual.dato["id"] == id_paciente:
                return actual.dato
            actual = actual.siguiente
        return None

    def buscar_por_nombre(self, nombre_paciente):
        nombre_normalizado = nombre_paciente.strip().lower()
        actual = self.frente
        while actual is not None:
            if actual.dato["paciente"].strip().lower() == nombre_normalizado:
                return actual.dato
            actual = actual.siguiente
        return None

    def eliminar_por_id(self, id_paciente):
        actual = self.frente
        anterior = None
        while actual is not None:
            if actual.dato["id"] == id_paciente:
                if anterior is None:
                    self.frente = actual.siguiente
                else:
                    anterior.siguiente = actual.siguiente
                if actual is self.final:
                    self.final = anterior
                self.tamano -= 1
                return True
            anterior = actual
            actual = actual.siguiente
        return False

    def eliminar_por_nombre(self, nombre_paciente):
        nombre_normalizado = nombre_paciente.strip().lower()
        actual = self.frente
        anterior = None
        while actual is not None:
            if actual.dato["paciente"].strip().lower() == nombre_normalizado:
                if anterior is None:
                    self.frente = actual.siguiente
                else:
                    anterior.siguiente = actual.siguiente
                if actual is self.final:
                    self.final = anterior
                self.tamano -= 1
                return actual.dato
            anterior = actual
            actual = actual.siguiente
        return None

    def vaciar(self):
        self.frente = None
        self.final = None
        self.tamano = 0

    def obtener_lista(self):
        resultado = []
        actual = self.frente
        while actual is not None:
            resultado.append(actual.dato)
            actual = actual.siguiente
        return resultado

    def __len__(self):
        return self.tamano


In [4]:
# Algoritmos de ordenamiento BubbleSort y QuickSort
import time
def bubble_sort(lista, key, descendente=False):
    arreglo = lista.copy()
    n = len(arreglo)
    for i in range(n - 1):
        hubo_intercambio = False
        for j in range(n - 1 - i):
            if descendente:
                debe_intercambiar = arreglo[j][key] < arreglo[j + 1][key]
            else:
                debe_intercambiar = arreglo[j][key] > arreglo[j + 1][key]
            if debe_intercambiar:
                arreglo[j], arreglo[j + 1] = arreglo[j + 1], arreglo[j]
                hubo_intercambio = True
        if not hubo_intercambio:
            break
    return arreglo

def _particionar(arreglo, inicio, fin, key, descendente=False):
    pivote = arreglo[fin][key]
    i = inicio - 1
    for j in range(inicio, fin):
        if descendente:
            cumple = arreglo[j][key] >= pivote
        else:
            cumple = arreglo[j][key] <= pivote
        if cumple:
            i += 1
            arreglo[i], arreglo[j] = arreglo[j], arreglo[i]
    arreglo[i + 1], arreglo[fin] = arreglo[fin], arreglo[i + 1]
    return i + 1

def _quick_sort_recursivo(arreglo, inicio, fin, key, descendente=False):
    if inicio < fin:
        posicion_pivote = _particionar(arreglo, inicio, fin, key, descendente)
        _quick_sort_recursivo(arreglo, inicio, posicion_pivote - 1, key, descendente)
        _quick_sort_recursivo(arreglo, posicion_pivote + 1, fin, key, descendente)

def quick_sort(lista, key, descendente=False):
    arreglo = lista.copy()
    _quick_sort_recursivo(arreglo, 0, len(arreglo) - 1, key, descendente)
    return arreglo

def medir_tiempo_ordenamiento(funcion_ordenamiento, lista, key, descendente=False):
    inicio = time.perf_counter()
    resultado = funcion_ordenamiento(lista, key, descendente)
    fin = time.perf_counter()
    milisegundos = (fin - inicio) * 1000
    return resultado, milisegundos


In [5]:
# Busqueda binaria
def busqueda_binaria(lista_ordenada, valor_buscado, key):
    izquierda = 0
    derecha = len(lista_ordenada) - 1
    while izquierda <= derecha:
        medio = (izquierda + derecha) // 2
        if lista_ordenada[medio][key] == valor_buscado:
            return medio
        elif lista_ordenada[medio][key] < valor_buscado:
            izquierda = medio + 1
        else:
            derecha = medio - 1
    return -1


In [6]:
# Escala de prioridad y generacion de datos
import random
PRIORIDADES = {
    1: {
        "etiqueta": "Leve",
        "descripcion": "Raspaduras, magulladuras pequenas, cortaduras sin puntos, dolor minimo.",
        "espera_min": 180,
        "espera_max": 240,
    },
    2: {
        "etiqueta": "Menor",
        "descripcion": "Esguinces leves, laceraciones menores con puntos, quemaduras de primer grado.",
        "espera_min": 90,
        "espera_max": 150,
    },
    3: {
        "etiqueta": "Moderado",
        "descripcion": "Fracturas simples, contusiones significativas con hematomas, laceraciones que requieren sutura.",
        "espera_min": 30,
        "espera_max": 60,
    },
    4: {
        "etiqueta": "Severo",
        "descripcion": "Fracturas expuestas, hemorragias significativas, lesiones internas, conmocion cerebral.",
        "espera_min": 5,
        "espera_max": 15,
    },
    5: {
        "etiqueta": "Critico / Riesgo vital",
        "descripcion": "Peligro vital inmediato, hemorragias incontrolables, paro cardiorrespiratorio.",
        "espera_min": 0,
        "espera_max": 1,
    },
}

def texto_prioridad(nivel):
    info = PRIORIDADES[nivel]
    return f"{nivel} - {info['etiqueta']}"

def texto_recomendacion(nivel):
    info = PRIORIDADES[nivel]
    if info["espera_max"] == 0:
        return f"Nivel {nivel} ({info['etiqueta']}): atencion inmediata (0 min de espera)."
    return (
        f"Nivel {nivel} ({info['etiqueta']}): tiempo de espera recomendado "
        f"entre {info['espera_min']} y {info['espera_max']} minutos."
    )

def tiempo_espera_sugerido(nivel):
    info = PRIORIDADES[nivel]
    return round((info["espera_min"] + info["espera_max"]) / 2, 2)

NOMBRES = ["Ana", "Luis", "Maria", "Carlos", "Sofia", "Pedro", "Lucia", "Jorge", "Elena", "Diego", "Valentina", "Andres", "Camila", "Mateo", "Paula"]
APELLIDOS = ["Perez", "Gomez", "Rodriguez", "Torres", "Ramirez", "Flores", "Castro", "Vega", "Morales", "Suarez"]

def generar_registro_aleatorio(id_registro):
    paciente = f"{random.choice(NOMBRES)} {random.choice(APELLIDOS)}"
    prioridad = random.randint(1, 5)
    info = PRIORIDADES[prioridad]
    limite_superior = max(info["espera_max"], info["espera_min"])
    tiempo_espera = round(random.uniform(info["espera_min"], limite_superior), 2)
    return {
        "id": id_registro,
        "paciente": paciente,
        "codigo": f"PAC-{random.randint(10000, 99999)}",
        "prioridad": prioridad,
        "tiempo_espera": tiempo_espera,
    }


In [7]:
# Dibujo del grafo para incrustar en la interfaz
import networkx as nx
def dibujar_grafo_en_eje(grafo, eje, titulo="Grafo Aciclico Dirigido", semilla=42):
    eje.clear()
    G = nx.DiGraph()
    for nodo in grafo.obtener_nodos():
        G.add_node(nodo)
    for nodo in grafo.obtener_nodos():
        for sucesor in grafo.obtener_sucesores(nodo):
            G.add_edge(nodo, sucesor)
    if G.number_of_nodes() == 0:
        eje.text(0.5, 0.5, "El grafo esta vacio", ha="center", va="center", fontsize=11)
    else:
        posiciones = nx.spring_layout(G, seed=semilla, k=1.2)
        nx.draw_networkx_nodes(G, posiciones, ax=eje, node_size=1400, node_color="white", edgecolors="black", linewidths=1.5)
        nx.draw_networkx_labels(G, posiciones, ax=eje, font_size=8, font_weight="bold")
        nx.draw_networkx_edges(G, posiciones, ax=eje, arrowstyle="-|>", arrowsize=18, node_size=1400, edge_color="black", width=1.3)
    eje.set_title(titulo)
    eje.axis("off")


In [8]:
# Interfaz grafica
import tkinter as tk
from tkinter import ttk, messagebox

import matplotlib
matplotlib.use("TkAgg")
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

class AppEmergencias:
    def __init__(self, root, grafo_emergencias):
        self.root = root
        self.root.title("Sistema de Gestion de Emergencias Hospitalarias")
        self.root.geometry("1150x720")

        self.cola_pacientes = Cola()
        self.grafo = grafo_emergencias
        self.contador_id = 1
        self.ids_usados = set()

        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill="both", expand=True, padx=10, pady=10)

        self.tab_cola = ttk.Frame(self.notebook)
        self.tab_grafo = ttk.Frame(self.notebook)
        self.notebook.add(self.tab_cola, text="Panel de Pacientes")
        self.notebook.add(self.tab_grafo, text="Proceso de Emergencias (DAG)")

        self._construir_tab_cola()
        self._construir_tab_grafo()

    # PANEL DE PACIENTES
    def _construir_tab_cola(self):
        # Registro de los pacientes
        frame_formulario = ttk.LabelFrame(self.tab_cola, text="Registro de pacientes")
        frame_formulario.pack(fill="x", padx=10, pady=5)

        ttk.Label(frame_formulario, text="ID:").grid(row=0, column=0, padx=5, pady=5, sticky="e")
        self.entry_id_manual = ttk.Entry(frame_formulario, width=8)
        self.entry_id_manual.grid(row=0, column=1, padx=5, pady=5)

        ttk.Label(frame_formulario, text="Paciente:").grid(row=0, column=2, padx=5, pady=5, sticky="e")
        self.entry_paciente = ttk.Entry(frame_formulario, width=20)
        self.entry_paciente.grid(row=0, column=3, padx=5, pady=5)

        ttk.Label(frame_formulario, text="Prioridad:").grid(row=0, column=4, padx=5, pady=5, sticky="e")
        self.combo_prioridad = ttk.Combobox(
            frame_formulario, width=22, state="readonly",
            values=[texto_prioridad(n) for n in range(1, 6)],
        )
        self.combo_prioridad.grid(row=0, column=5, padx=5, pady=5)
        self.combo_prioridad.bind("<<ComboboxSelected>>", self.actualizar_recomendacion_espera)

        ttk.Label(frame_formulario, text="Tiempo de espera (min):").grid(row=1, column=4, padx=5, pady=5, sticky="e")
        self.entry_tiempo = ttk.Entry(frame_formulario, width=10)
        self.entry_tiempo.grid(row=1, column=5, padx=5, pady=5)

        self.label_recomendacion = ttk.Label(
            frame_formulario, text="Seleccione una prioridad para ver el tiempo de espera recomendado.",
            foreground="#1e73a7",
        )
        self.label_recomendacion.grid(row=1, column=0, columnspan=4, padx=5, pady=5, sticky="w")

        ttk.Button(frame_formulario, text="Agregar paciente", command=self.agregar_paciente_manual).grid(row=0, column=6, rowspan=2, padx=10, pady=5)

        # Busqueda
        frame_busqueda = ttk.LabelFrame(self.tab_cola, text="Busqueda")
        frame_busqueda.pack(fill="x", padx=10, pady=5)

        ttk.Label(frame_busqueda, text="ID:").grid(row=0, column=0, padx=5, pady=5)
        self.entry_id_buscar = ttk.Entry(frame_busqueda, width=10)
        self.entry_id_buscar.grid(row=0, column=1, padx=5, pady=5)
        ttk.Button(frame_busqueda, text="Buscar", command=self.buscar_por_id).grid(row=0, column=2, padx=5, pady=5)

        ttk.Label(frame_busqueda, text="Nombre:").grid(row=0, column=3, padx=5, pady=5)
        self.entry_nombre_buscar = ttk.Entry(frame_busqueda, width=18)
        self.entry_nombre_buscar.grid(row=0, column=4, padx=5, pady=5)
        ttk.Button(frame_busqueda, text="Buscar", command=self.buscar_por_nombre).grid(row=0, column=5, padx=5, pady=5)

        # Eliminacion y carga masiva
        frame_acciones = ttk.LabelFrame(self.tab_cola, text="Eliminacion/Carga masiva")
        frame_acciones.pack(fill="x", padx=10, pady=5)

        ttk.Label(frame_acciones, text="Eliminar por ID:").grid(row=0, column=0, padx=5, pady=5)
        self.entry_id_eliminar = ttk.Entry(frame_acciones, width=10)
        self.entry_id_eliminar.grid(row=0, column=1, padx=5, pady=5)

        ttk.Label(frame_acciones, text="Eliminar por nombre:").grid(row=0, column=2, padx=5, pady=5)
        self.entry_nombre_eliminar = ttk.Entry(frame_acciones, width=18)
        self.entry_nombre_eliminar.grid(row=0, column=3, padx=5, pady=5)

        ttk.Button(frame_acciones, text="Eliminar", command=self.eliminar_paciente).grid(row=0, column=4, padx=5, pady=5)
        ttk.Button(frame_acciones, text="Eliminar todos los registros", command=self.eliminar_todos).grid(row=0, column=5, padx=5, pady=5)
        ttk.Button(frame_acciones, text="Generar 1000 aleatorios", command=self.generar_carga_masiva).grid(row=0, column=6, padx=5, pady=5)

        # Ordenamiento
        frame_orden = ttk.LabelFrame(self.tab_cola, text="Ordenamiento")
        frame_orden.pack(fill="x", padx=10, pady=5)
        ttk.Button(frame_orden, text="Ordenar - BubbleSort por prioridad", command=lambda: self.ordenar_datos("bubble")).grid(row=0, column=0, padx=5, pady=5)
        ttk.Button(frame_orden, text="Ordenar - QuickSort por tiempo de espera", command=lambda: self.ordenar_datos("quick")).grid(row=0, column=1, padx=5, pady=5)
        self.label_tiempo = ttk.Label(frame_orden, text="Tiempo de ejecucion: -- ms")
        self.label_tiempo.grid(row=0, column=2, padx=15, pady=5)

        # Tabla de pacientes
        columnas = ("id", "paciente", "codigo", "prioridad", "tiempo_espera")
        encabezados = ("ID", "Paciente", "Codigo", "Prioridad", "Tiempo espera (min)")
        self.tabla_pacientes = ttk.Treeview(self.tab_cola, columns=columnas, show="headings", height=13)
        for columna, encabezado in zip(columnas, encabezados):
            self.tabla_pacientes.heading(columna, text=encabezado)
            self.tabla_pacientes.column(columna, width=150, anchor="center")
        self.tabla_pacientes.pack(fill="both", expand=True, padx=10, pady=5)

        self.label_conteo = ttk.Label(self.tab_cola, text="Total de pacientes en cola: 0")
        self.label_conteo.pack(anchor="w", padx=10)

    def actualizar_recomendacion_espera(self, evento=None):
        nivel = self._nivel_prioridad_seleccionado()
        if nivel is None:
            return
        self.label_recomendacion.config(text=texto_recomendacion(nivel))
        self.entry_tiempo.delete(0, tk.END)
        self.entry_tiempo.insert(0, str(tiempo_espera_sugerido(nivel)))

    def _nivel_prioridad_seleccionado(self):
        valor = self.combo_prioridad.get()
        if not valor:
            return None
        return int(valor.split(" - ")[0])

    def agregar_paciente_manual(self):
        id_texto = self.entry_id_manual.get().strip()
        paciente = self.entry_paciente.get().strip()
        tiempo_texto = self.entry_tiempo.get().strip()
        nivel = self._nivel_prioridad_seleccionado()

        if not id_texto or not paciente or nivel is None or not tiempo_texto:
            messagebox.showwarning("Datos incompletos", "Complete el ID, el paciente, la prioridad y el tiempo de espera.")
            return
        if not id_texto.isdigit():
            messagebox.showerror("Error de formato", "El ID debe ser un numero entero.")
            return

        id_paciente = int(id_texto)
        if id_paciente in self.ids_usados:
            messagebox.showerror("ID duplicado", f"El ID {id_paciente} ya esta en uso. Ingrese un ID distinto.")
            return

        try:
            tiempo_espera = float(tiempo_texto)
        except ValueError:
            messagebox.showerror("Error de formato", "El tiempo de espera debe ser numerico.")
            return

        registro = {
            "id": id_paciente,
            "paciente": paciente,
            "codigo": f"PAC-{id_paciente:05d}",
            "prioridad": nivel,
            "tiempo_espera": tiempo_espera,
        }
        self.cola_pacientes.encolar(registro)
        self.ids_usados.add(id_paciente)
        self.contador_id = max(self.contador_id, id_paciente + 1)

        info_prioridad = PRIORIDADES[nivel]
        messagebox.showinfo(
            "Paciente registrado",
            f"Paciente registrado.\n\n"
            f"ID: {registro['id']}\n"
            f"Paciente: {registro['paciente']}\n"
            f"Codigo: {registro['codigo']}\n"
            f"Prioridad: {registro['prioridad']} - {info_prioridad['etiqueta']}\n"
            f"Tiempo de espera: {registro['tiempo_espera']} min",
        )

        self.entry_id_manual.delete(0, tk.END)
        self.entry_paciente.delete(0, tk.END)
        self.entry_tiempo.delete(0, tk.END)
        self.combo_prioridad.set("")
        self.label_recomendacion.config(text="Seleccione una prioridad para ver el tiempo de espera recomendado.")
        self.actualizar_tabla(self.cola_pacientes.obtener_lista())

    def _siguiente_id_automatico(self):
        while self.contador_id in self.ids_usados:
            self.contador_id += 1
        id_generado = self.contador_id
        self.contador_id += 1
        return id_generado

    def generar_carga_masiva(self):
        cantidad = 1000
        for _ in range(cantidad):
            id_generado = self._siguiente_id_automatico()
            registro = generar_registro_aleatorio(id_generado)
            self.cola_pacientes.encolar(registro)
            self.ids_usados.add(id_generado)
        self.actualizar_tabla(self.cola_pacientes.obtener_lista())
        messagebox.showinfo("Carga masiva completa", f"Se generaron {cantidad} registros aleatorios.")

    def buscar_por_id(self):
        id_texto = self.entry_id_buscar.get().strip()
        if not id_texto.isdigit():
            messagebox.showerror("Error", "Ingrese un ID numerico valido.")
            return
        lista_ordenada = quick_sort(self.cola_pacientes.obtener_lista(), "id")
        posicion = busqueda_binaria(lista_ordenada, int(id_texto), "id")
        if posicion == -1:
            messagebox.showwarning("No encontrado", f"No existe un paciente con ID {id_texto}.")
            return
        self._mostrar_info_paciente(lista_ordenada[posicion])

    def buscar_por_nombre(self):
        nombre = self.entry_nombre_buscar.get().strip()
        if not nombre:
            messagebox.showwarning("Datos incompletos", "Ingrese el nombre a buscar.")
            return
        registro = self.cola_pacientes.buscar_por_nombre(nombre)
        if registro is None:
            messagebox.showwarning("No encontrado", f"No existe un paciente con nombre '{nombre}'.")
            return
        self._mostrar_info_paciente(registro)

    def _mostrar_info_paciente(self, registro):
        info_prioridad = PRIORIDADES[registro["prioridad"]]
        messagebox.showinfo(
            "Paciente encontrado",
            f"ID: {registro['id']}\n"
            f"Paciente: {registro['paciente']}\n"
            f"Codigo: {registro['codigo']}\n"
            f"Prioridad: {registro['prioridad']} - {info_prioridad['etiqueta']}\n"
            f"Tiempo de espera: {registro['tiempo_espera']} min",
        )

    def eliminar_paciente(self):
        id_texto = self.entry_id_eliminar.get().strip()
        nombre_texto = self.entry_nombre_eliminar.get().strip()

        if id_texto:
            if not id_texto.isdigit():
                messagebox.showerror("Error", "Ingrese un ID numerico valido.")
                return
            id_paciente = int(id_texto)
            eliminado = self.cola_pacientes.eliminar_por_id(id_paciente)
            if eliminado:
                self.ids_usados.discard(id_paciente)
                self.entry_id_eliminar.delete(0, tk.END)
                self.actualizar_tabla(self.cola_pacientes.obtener_lista())
                messagebox.showinfo("Eliminado", f"Paciente con ID {id_paciente} eliminado.")
            else:
                messagebox.showwarning("No encontrado", f"No existe un paciente con ID {id_paciente}.")
            return

        if nombre_texto:
            registro_eliminado = self.cola_pacientes.eliminar_por_nombre(nombre_texto)
            if registro_eliminado:
                self.ids_usados.discard(registro_eliminado["id"])
                self.entry_nombre_eliminar.delete(0, tk.END)
                self.actualizar_tabla(self.cola_pacientes.obtener_lista())
                messagebox.showinfo("Eliminado", f"Paciente '{nombre_texto}' eliminado.")
            else:
                messagebox.showwarning("No encontrado", f"No existe un paciente con nombre '{nombre_texto}'.")
            return

        messagebox.showwarning("Datos incompletos", "Ingrese un ID o un nombre para eliminar.")

    def eliminar_todos(self):
        if len(self.cola_pacientes) == 0:
            messagebox.showinfo("Sin registros", "No hay pacientes registrados.")
            return
        confirmar = messagebox.askyesno(
            "Confirmar eliminacion",
            f"Se eliminaran los {len(self.cola_pacientes)} registros de la cola. Esta accion no se puede deshacer. Continuar?",
        )
        if not confirmar:
            return
        self.cola_pacientes.vaciar()
        self.ids_usados.clear()
        self.contador_id = 1
        self.actualizar_tabla(self.cola_pacientes.obtener_lista())
        messagebox.showinfo("Registros eliminados", "Se eliminaron todos los registros de la cola.")

    def ordenar_datos(self, algoritmo):
        lista = self.cola_pacientes.obtener_lista()
        if not lista:
            messagebox.showwarning("Sin datos", "No hay pacientes registrados para ordenar.")
            return
        if algoritmo == "bubble":
            resultado, tiempo_ms = medir_tiempo_ordenamiento(bubble_sort, lista, "prioridad", descendente=True)
            nombre_algoritmo = "BubbleSort (por prioridad, de 5 a 1)"
        else:
            resultado, tiempo_ms = medir_tiempo_ordenamiento(quick_sort, lista, "tiempo_espera", descendente=False)
            nombre_algoritmo = "QuickSort (por tiempo de espera, menor a mayor)"
        self.actualizar_tabla(resultado)
        self.label_tiempo.config(text=f"{nombre_algoritmo}: {tiempo_ms:.4f} ms para {len(lista)} registros")

    def actualizar_tabla(self, lista_registros, limite_visual=500):
        self.tabla_pacientes.delete(*self.tabla_pacientes.get_children())
        for registro in lista_registros[:limite_visual]:
            info_prioridad = PRIORIDADES[registro["prioridad"]]
            self.tabla_pacientes.insert(
                "", "end",
                values=(registro["id"], registro["paciente"], registro["codigo"],
                        f"{registro['prioridad']} - {info_prioridad['etiqueta']}",
                        registro["tiempo_espera"]),
            )
        total = len(lista_registros)
        texto_extra = f" (mostrando los primeros {limite_visual})" if total > limite_visual else ""
        self.label_conteo.config(text=f"Total de pacientes en cola: {total}{texto_extra}")

    # GRAFO DE EMERGENCIAS
    def _construir_tab_grafo(self):
        frame_arista = ttk.LabelFrame(self.tab_grafo, text="Registro manual de dependencia (arista)")
        frame_arista.pack(fill="x", padx=10, pady=5)

        etapas_proceso = [
            "Recepcion del paciente",
            "Triaje",
            "Diagnostico",
            "Solicitar examenes",
            "Estabilizacion",
            "Tratamiento",
            "Informe clinico",
            "Alta medica",
        ]
        self.etapas_proceso = etapas_proceso

        ttk.Label(frame_arista, text="Etapa origen:").grid(row=0, column=0, padx=5, pady=5)
        self.entry_origen = ttk.Combobox(frame_arista, width=23, state="readonly", values=etapas_proceso)
        self.entry_origen.grid(row=0, column=1, padx=5, pady=5)
        ttk.Label(frame_arista, text="Etapa destino:").grid(row=0, column=2, padx=5, pady=5)
        self.entry_destino = ttk.Combobox(frame_arista, width=23, state="readonly", values=etapas_proceso)
        self.entry_destino.grid(row=0, column=3, padx=5, pady=5)
        ttk.Button(frame_arista, text="Agregar arista", command=self.agregar_arista_manual).grid(row=0, column=4, padx=5, pady=5)

        frame_nodo = ttk.LabelFrame(self.tab_grafo, text="Busqueda y eliminacion de etapas (nodos)")
        frame_nodo.pack(fill="x", padx=10, pady=5)
        ttk.Label(frame_nodo, text="Nombre de etapa:").grid(row=0, column=0, padx=5, pady=5)
        self.entry_nodo = ttk.Combobox(frame_nodo, width=23, state="readonly", values=etapas_proceso)
        self.entry_nodo.grid(row=0, column=1, padx=5, pady=5)
        ttk.Button(frame_nodo, text="Buscar etapa", command=self.buscar_nodo_gui).grid(row=0, column=2, padx=5, pady=5)
        ttk.Button(frame_nodo, text="Eliminar etapa", command=self.eliminar_nodo_gui).grid(row=0, column=3, padx=5, pady=5)

        frame_acciones = ttk.Frame(self.tab_grafo)
        frame_acciones.pack(fill="x", padx=10, pady=5)
        ttk.Button(frame_acciones, text="Calcular orden topologico", command=self.mostrar_orden_topologico).pack(side="left", padx=5)
        ttk.Button(frame_acciones, text="Dibujar grafo", command=self.dibujar_grafo_gui).pack(side="left", padx=5)
        frame_canvas = ttk.LabelFrame(self.tab_grafo, text="Visualizacion del grafo")
        frame_canvas.pack(fill="both", expand=True, padx=10, pady=5)

        self.figura_grafo = Figure(figsize=(6.5, 5.2), dpi=90)
        self.ejes_grafo = self.figura_grafo.add_subplot(111)
        self.canvas_grafo = FigureCanvasTkAgg(self.figura_grafo, master=frame_canvas)
        self.canvas_grafo.get_tk_widget().pack(fill="both", expand=True, padx=5, pady=5)

        self.dibujar_grafo_gui()

    def agregar_arista_manual(self):
        origen = self.entry_origen.get().strip()
        destino = self.entry_destino.get().strip()
        if not origen or not destino:
            messagebox.showwarning("Datos incompletos", "Ingrese la etapa de origen y destino.")
            return
        self.grafo.agregar_arista(origen, destino)
        self.entry_origen.delete(0, tk.END)
        self.entry_destino.delete(0, tk.END)
        self.dibujar_grafo_gui()

    def buscar_nodo_gui(self):
        nombre = self.entry_nodo.get().strip()
        if not nombre:
            messagebox.showwarning("Datos incompletos", "Ingrese el nombre de la etapa a buscar.")
            return
        existe = self.grafo.buscar_nodo(nombre)
        if existe:
            sucesores = self.grafo.obtener_sucesores(nombre)
            texto = f"La etapa '{nombre}' existe en el grafo."
            if sucesores:
                texto += f"\nDepende de ella: {', '.join(str(s) for s in sucesores)}"
            messagebox.showinfo("Etapa encontrada", texto)
        else:
            messagebox.showwarning("No encontrada", f"La etapa '{nombre}' no existe en el grafo.")

    def eliminar_nodo_gui(self):
        nombre = self.entry_nodo.get().strip()
        if not nombre:
            messagebox.showwarning("Datos incompletos", "Ingrese el nombre de la etapa a eliminar.")
            return
        eliminado = self.grafo.eliminar_nodo(nombre)
        if eliminado:
            self.entry_nodo.delete(0, tk.END)
            self.dibujar_grafo_gui()
            messagebox.showinfo("Eliminada", f"La etapa '{nombre}' fue eliminada del grafo.")
        else:
            messagebox.showwarning("No encontrada", f"La etapa '{nombre}' no existe en el grafo.")

    def mostrar_orden_topologico(self):
        orden, es_valido = ordenamiento_topologico(self.grafo)
        if es_valido:
            texto = "Orden de ejecucion sugerido:\n\n" + "\n".join(
                f"{posicion}. {tarea}" for posicion, tarea in enumerate(orden, start=1)
            )
            messagebox.showinfo("Orden topologico", texto)
        else:
            messagebox.showerror("Ciclo detectado", "El grafo contiene un ciclo: no es posible generar un orden valido.")

    def dibujar_grafo_gui(self):
        dibujar_grafo_en_eje(self.grafo, self.ejes_grafo, titulo="Sistema de Gestion de Emergencias (DAG)")
        self.canvas_grafo.draw()


In [9]:
# Grafo base del proceso de atencion
sistema_emergencias = GrafoAciclico()
sistema_emergencias.agregar_arista("Recepcion del paciente", "Triaje")
sistema_emergencias.agregar_arista("Triaje", "Diagnostico")
sistema_emergencias.agregar_arista("Diagnostico", "Solicitar examenes")
sistema_emergencias.agregar_arista("Diagnostico", "Estabilizacion")
sistema_emergencias.agregar_arista("Diagnostico", "Informe clinico")
sistema_emergencias.agregar_arista("Solicitar examenes", "Tratamiento")
sistema_emergencias.agregar_arista("Estabilizacion", "Tratamiento")
sistema_emergencias.agregar_arista("Tratamiento", "Informe clinico")
sistema_emergencias.agregar_arista("Informe clinico", "Alta medica")
sistema_emergencias.mostrar_grafo()


Estructura del grafo (lista de adyacencia):
  Recepcion del paciente -> Triaje
  Triaje -> Diagnostico
  Diagnostico -> Solicitar examenes, Estabilizacion, Informe clinico
  Solicitar examenes -> Tratamiento
  Estabilizacion -> Tratamiento
  Informe clinico -> Alta medica
  Tratamiento -> Informe clinico
  Alta medica -> (sin dependientes)


In [ ]:
# Ejecucion de la aplicacion
if __name__ == "__main__":
    ventana = tk.Tk()
    aplicacion = AppEmergencias(ventana, sistema_emergencias)
    ventana.mainloop()
